# Modeling - Diabetes Prediction

## Objective
Build, tune, and evaluate multiple models for diabetes prediction.

## Models
1. Logistic Regression (baseline)
2. Random Forest
3. XGBoost
4. LightGBM

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, precision_score, recall_score, 
                             f1_score, accuracy_score, roc_curve, 
                             confusion_matrix, classification_report)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
import xgboost as xgb
import lightgbm as lgb
import optuna
import joblib
import json
import os

optuna.logging.set_verbosity(optuna.logging.WARNING)

# Create directories
os.makedirs('../artifacts/models', exist_ok=True)
os.makedirs('../artifacts/predictions', exist_ok=True)
os.makedirs('../reports/tables', exist_ok=True)
os.makedirs('../reports/figures', exist_ok=True)

# Set random seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Libraries loaded successfully')

## 1. Data Loading & Preprocessing

In [ ]:
# Load data
df = pd.read_csv('../data/raw/diabetes-prediction-dataset.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nTarget distribution:')
print(df['diabetes'].value_counts(normalize=True))

In [ ]:
# Handle smoking_history 'No Info' - map to 'never' as reasonable assumption
df['smoking_history'] = df['smoking_history'].replace('No Info', 'never')
print('Smoking history after mapping:')
print(df['smoking_history'].value_counts())

In [ ]:
# Define feature types
categorical_features = ['gender', 'smoking_history']
binary_features = ['hypertension', 'heart_disease']
numerical_features = ['age', 'bmi', 'HbA1c_level', 'blood_glucose_level']

print(f'Categorical features: {categorical_features}')
print(f'Binary features: {binary_features}')
print(f'Numerical features: {numerical_features}')

In [ ]:
# Create preprocessing pipeline
# Ordinal mapping for smoking_history
smoking_order = {'never': 0, 'not current': 1, 'former': 2, 'current': 3, 'ever': 4}
df['smoking_history_encoded'] = df['smoking_history'].map(smoking_order)

# Gender encoding (N-1 dummies)
df['gender_male'] = (df['gender'] == 'Male').astype(int)

# Prepare final feature set
feature_columns = numerical_features + binary_features + ['gender_male', 'smoking_history_encoded']
X = df[feature_columns].copy()
y = df['diabetes'].copy()

print(f'Feature matrix shape: {X.shape}')
print(f'Features: {X.columns.tolist()}')

## 2. Train/Validation/Test Split

In [ ]:
# First split: 70% train, 30% temp (validation + test)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y
)

# Second split: 15% validation, 15% test (from the 30% temp)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=RANDOM_STATE, stratify=y_temp
)

print(f'Train set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.1f}%)')
print(f'Validation set: {X_val.shape[0]} samples ({X_val.shape[0]/len(X)*100:.1f}%)')
print(f'Test set: {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.1f}%)')
print(f'\nTarget distribution per split:')
print(f'Train diabetes rate: {y_train.mean():.4f}')
print(f'Validation diabetes rate: {y_val.mean():.4f}')
print(f'Test diabetes rate: {y_test.mean():.4f}')

In [ ]:
# Scale numerical features
scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_val_scaled = X_val.copy()
X_test_scaled = X_test.copy()

X_train_scaled[numerical_features] = scaler.fit_transform(X_train[numerical_features])
X_val_scaled[numerical_features] = scaler.transform(X_val[numerical_features])
X_test_scaled[numerical_features] = scaler.transform(X_test[numerical_features])

print('Features scaled successfully.')
print(f'\nTrain set statistics (scaled):')
print(X_train_scaled[numerical_features].describe().loc[['mean', 'std']].round(4))

## 3. Model Training & Evaluation Functions

In [ ]:
def evaluate_model(model, X_train, X_val, X_test, y_train, y_val, y_test, model_name, threshold=0.5):
    """Evaluate model on train, validation, and test sets."""
    results = {}
    
    for split_name, X_split, y_split in [('train', X_train, y_train), 
                                           ('val', X_val, y_val), 
                                           ('test', X_test, y_test)]:
        y_pred_proba = model.predict_proba(X_split)[:, 1]
        y_pred = (y_pred_proba >= threshold).astype(int)
        
        results[split_name] = {
            'roc_auc': roc_auc_score(y_split, y_pred_proba),
            'precision': precision_score(y_split, y_pred),
            'recall': recall_score(y_split, y_pred),
            'f1': f1_score(y_split, y_pred),
            'accuracy': accuracy_score(y_split, y_pred)
        }
    
    return results

def print_results(results, model_name):
    """Print model results in a formatted table."""
    print(f'\n{"="*60}')
    print(f'{model_name} Results')
    print(f'{"="*60}')
    
    for split in ['train', 'val', 'test']:
        print(f'\n{split.upper()} Set:')
        for metric, value in results[split].items():
            print(f'  {metric:>12}: {value:.4f}')
    
    print(f'\nOverfitting check (val - train ROC-AUC): {results["val"]["roc_auc"] - results["train"]["roc_auc"]:.4f}')

## 4. Baseline Model - Logistic Regression

In [ ]:
# Logistic Regression baseline
lr_model = LogisticRegression(
    random_state=RANDOM_STATE,
    max_iter=1000,
    class_weight='balanced'
)
lr_model.fit(X_train_scaled, y_train)

lr_results = evaluate_model(lr_model, X_train_scaled, X_val_scaled, X_test_scaled, 
                             y_train, y_val, y_test, 'Logistic Regression')
print_results(lr_results, 'Logistic Regression')

## 5. Hyperparameter Tuning with Optuna

In [ ]:
def objective_logistic(trial):
    """Optuna objective for Logistic Regression."""
    params = {
        'C': trial.suggest_float('C', 1e-4, 100, log=True),
        'penalty': trial.suggest_categorical('penalty', ['l1', 'l2']),
        'solver': 'saga'
    }
    
    model = LogisticRegression(**params, random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced')
    model.fit(X_train_scaled, y_train)
    y_pred_proba = model.predict_proba(X_val_scaled)[:, 1]
    return roc_auc_score(y_val, y_pred_proba)

# Optimize
study_lr = optuna.create_study(direction='maximize')
study_lr.optimize(objective_logistic, n_trials=50, show_progress_bar=True)

print(f'\nBest LR ROC-AUC: {study_lr.best_value:.4f}')
print(f'Best LR params: {study_lr.best_params}')

In [ ]:
# Train best Logistic Regression
lr_best = LogisticRegression(**study_lr.best_params, solver='saga', 
                             random_state=RANDOM_STATE, max_iter=1000, class_weight='balanced')
lr_best.fit(X_train_scaled, y_train)

lr_best_results = evaluate_model(lr_best, X_train_scaled, X_val_scaled, X_test_scaled,
                                  y_train, y_val, y_test, 'Logistic Regression (Tuned)')
print_results(lr_best_results, 'Logistic Regression (Tuned)')

In [ ]:
def objective_rf(trial):
    """Optuna objective for Random Forest."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 5, 30),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', None])
    }
    
    model = RandomForestClassifier(**params, random_state=RANDOM_STATE, class_weight='balanced', n_jobs=-1)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred_proba)

study_rf = optuna.create_study(direction='maximize')
study_rf.optimize(objective_rf, n_trials=50, show_progress_bar=True)

print(f'\nBest RF ROC-AUC: {study_rf.best_value:.4f}')
print(f'Best RF params: {study_rf.best_params}')

In [ ]:
# Train best Random Forest
rf_best = RandomForestClassifier(**study_rf.best_params, random_state=RANDOM_STATE, 
                                  class_weight='balanced', n_jobs=-1)
rf_best.fit(X_train, y_train)

rf_best_results = evaluate_model(rf_best, X_train, X_val, X_test,
                                  y_train, y_val, y_test, 'Random Forest (Tuned)')
print_results(rf_best_results, 'Random Forest (Tuned)')

In [ ]:
def objective_xgb(trial):
    """Optuna objective for XGBoost."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0)
    }
    
    model = xgb.XGBClassifier(**params, random_state=RANDOM_STATE, 
                               scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
                               eval_metric='logloss', use_label_encoder=False)
    model.fit(X_train, y_train, verbose=False)
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred_proba)

study_xgb = optuna.create_study(direction='maximize')
study_xgb.optimize(objective_xgb, n_trials=50, show_progress_bar=True)

print(f'\nBest XGB ROC-AUC: {study_xgb.best_value:.4f}')
print(f'Best XGB params: {study_xgb.best_params}')

In [ ]:
# Train best XGBoost
xgb_best = xgb.XGBClassifier(**study_xgb.best_params, random_state=RANDOM_STATE,
                               scale_pos_weight=len(y_train[y_train==0])/len(y_train[y_train==1]),
                               eval_metric='logloss', use_label_encoder=False)
xgb_best.fit(X_train, y_train, verbose=False)

xgb_best_results = evaluate_model(xgb_best, X_train, X_val, X_test,
                                    y_train, y_val, y_test, 'XGBoost (Tuned)')
print_results(xgb_best_results, 'XGBoost (Tuned)')

In [ ]:
def objective_lgb(trial):
    """Optuna objective for LightGBM."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 15),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0)
    }
    
    model = lgb.LGBMClassifier(**params, random_state=RANDOM_STATE, 
                                is_unbalance=True, verbose=-1)
    model.fit(X_train, y_train)
    y_pred_proba = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_val, y_pred_proba)

study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lgb, n_trials=50, show_progress_bar=True)

print(f'\nBest LGB ROC-AUC: {study_lgb.best_value:.4f}')
print(f'Best LGB params: {study_lgb.best_params}')

In [ ]:
# Train best LightGBM
lgb_best = lgb.LGBMClassifier(**study_lgb.best_params, random_state=RANDOM_STATE,
                               is_unbalance=True, verbose=-1)
lgb_best.fit(X_train, y_train)

lgb_best_results = evaluate_model(lgb_best, X_train, X_val, X_test,
                                    y_train, y_val, y_test, 'LightGBM (Tuned)')
print_results(lgb_best_results, 'LightGBM (Tuned)')

## 6. Model Comparison

In [ ]:
# Create comparison table
all_results = {
    'Logistic Regression': lr_best_results,
    'Random Forest': rf_best_results,
    'XGBoost': xgb_best_results,
    'LightGBM': lgb_best_results
}

comparison_rows = []
for model_name, results in all_results.items():
    for split in ['train', 'val', 'test']:
        row = {'Model': model_name, 'Split': split}
        row.update(results[split])
        comparison_rows.append(row)

comparison_df = pd.DataFrame(comparison_rows)
print('\nModel Comparison Table:')
print(comparison_df.round(4).to_string(index=False))

# Save to CSV
comparison_df.to_csv('../reports/tables/model_comparison.csv', index=False)
print('\nComparison table saved to reports/tables/model_comparison.csv')

In [ ]:
# Visual comparison - ROC-AUC
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart for ROC-AUC
models = list(all_results.keys())
val_aucs = [all_results[m]['val']['roc_auc'] for m in models]
test_aucs = [all_results[m]['test']['roc_auc'] for m in models]

x = np.arange(len(models))
width = 0.35

axes[0].bar(x - width/2, val_aucs, width, label='Validation', color='#3498db', edgecolor='black')
axes[0].bar(x + width/2, test_aucs, width, label='Test', color='#e74c3c', edgecolor='black')
axes[0].set_xlabel('Model', fontsize=12)
axes[0].set_ylabel('ROC-AUC', fontsize=12)
axes[0].set_title('ROC-AUC by Model', fontsize=14, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=45, ha='right')
axes[0].legend()
axes[0].set_ylim([0.8, 1.0])

for i, (v, t) in enumerate(zip(val_aucs, test_aucs)):
    axes[0].text(i - width/2, v + 0.002, f'{v:.3f}', ha='center', fontsize=10)
    axes[0].text(i + width/2, t + 0.002, f'{t:.3f}', ha='center', fontsize=10)

# Bar chart for F1
val_f1s = [all_results[m]['val']['f1'] for m in models]
test_f1s = [all_results[m]['test']['f1'] for m in models]

axes[1].bar(x - width/2, val_f1s, width, label='Validation', color='#3498db', edgecolor='black')
axes[1].bar(x + width/2, test_f1s, width, label='Test', color='#e74c3c', edgecolor='black')
axes[1].set_xlabel('Model', fontsize=12)
axes[1].set_ylabel('F1 Score', fontsize=12)
axes[1].set_title('F1 Score by Model', fontsize=14, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=45, ha='right')
axes[1].legend()

for i, (v, t) in enumerate(zip(val_f1s, test_f1s)):
    axes[1].text(i - width/2, v + 0.005, f'{v:.3f}', ha='center', fontsize=10)
    axes[1].text(i + width/2, t + 0.005, f'{t:.3f}', ha='center', fontsize=10)

plt.suptitle('Model Performance Comparison', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Threshold Analysis

In [ ]:
# Use the best model for threshold analysis (LightGBM)
best_model = lgb_best
model_name = 'LightGBM'

# Get predictions on validation set
y_val_proba = best_model.predict_proba(X_val)[:, 1]

# Test different thresholds
thresholds = np.arange(0.1, 0.9, 0.05)
threshold_results = []

for thresh in thresholds:
    y_pred = (y_val_proba >= thresh).astype(int)
    threshold_results.append({
        'threshold': thresh,
        'precision': precision_score(y_val, y_pred),
        'recall': recall_score(y_val, y_pred),
        'f1': f1_score(y_val, y_pred)
    })

thresh_df = pd.DataFrame(threshold_results)

# Plot threshold analysis
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(thresh_df['threshold'], thresh_df['precision'], marker='o', label='Precision', linewidth=2)
ax.plot(thresh_df['threshold'], thresh_df['recall'], marker='s', label='Recall', linewidth=2)
ax.plot(thresh_df['threshold'], thresh_df['f1'], marker='^', label='F1', linewidth=2)
ax.axvline(x=0.5, color='gray', linestyle='--', alpha=0.5, label='Default threshold (0.5)')
ax.set_xlabel('Threshold', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'{model_name} - Threshold Analysis (Validation Set)', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/threshold_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nThreshold Analysis Results:')
print(thresh_df.round(4).to_string(index=False))

In [ ]:
# Select threshold based on recall >= 0.60 requirement
target_recall = 0.60
valid_thresholds = thresh_df[thresh_df['recall'] >= target_recall]

if len(valid_thresholds) > 0:
    # Among thresholds meeting recall target, choose the one with highest F1
    best_threshold_row = valid_thresholds.loc[valid_thresholds['f1'].idxmax()]
    selected_threshold = best_threshold_row['threshold']
else:
    selected_threshold = 0.5

print(f'Selected threshold: {selected_threshold:.2f}')
print(f'Target recall: {target_recall:.2f}')
print(f'Achieved recall: {best_threshold_row["recall"]:.4f}')
print(f'Achieved precision: {best_threshold_row["precision"]:.4f}')
print(f'Achieved F1: {best_threshold_row["f1"]:.4f}')

## 8. Final Model Evaluation

In [ ]:
# Final evaluation with selected threshold
final_results = evaluate_model(best_model, X_train, X_val, X_test,
                               y_train, y_val, y_test, 'Final LightGBM', 
                               threshold=selected_threshold)

print(f'\nFinal Model: {model_name}')
print(f'Selected threshold: {selected_threshold:.2f}')
print_results(final_results, 'Final LightGBM')

In [ ]:
# Confusion matrix on test set
y_test_proba = best_model.predict_proba(X_test)[:, 1]
y_test_pred = (y_test_proba >= selected_threshold).astype(int)

cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['No Diabetes', 'Diabetes'],
            yticklabels=['No Diabetes', 'Diabetes'])
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title(f'Confusion Matrix - Test Set (Threshold={selected_threshold:.2f})', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/confusion_matrix_test.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nClassification Report (Test Set):')
print(classification_report(y_test, y_test_pred, target_names=['No Diabetes', 'Diabetes']))

In [ ]:
# ROC curves for all models
fig, ax = plt.subplots(figsize=(10, 8))

models_dict = {
    'Logistic Regression': (lr_best, X_test_scaled),
    'Random Forest': (rf_best, X_test),
    'XGBoost': (xgb_best, X_test),
    'LightGBM': (lgb_best, X_test)
}

colors = ['#3498db', '#2ecc71', '#e67e22', '#e74c3c']

for (name, (model, X_data)), color in zip(models_dict.items(), colors):
    y_proba = model.predict_proba(X_data)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {auc:.4f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curves - All Models (Test Set)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../reports/figures/roc_curves_all_models.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Feature Importance

In [ ]:
# Feature importance from LightGBM
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': best_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=feature_importance, x='importance', y='feature', palette='viridis', ax=ax)
ax.set_title(f'Feature Importance - {model_name}', fontsize=14, fontweight='bold')
ax.set_xlabel('Importance', fontsize=12)
ax.set_ylabel('Feature', fontsize=12)

for i, v in enumerate(feature_importance['importance']):
    ax.text(v + 0.5, i, f'{v:,}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nFeature Importance Ranking:')
print(feature_importance.to_string(index=False))

## 10. Save Model & Predictions

In [ ]:
# Save the best model
model_artifact = {
    'model': best_model,
    'scaler': scaler,
    'threshold': selected_threshold,
    'feature_columns': feature_columns,
    'numerical_features': numerical_features,
    'smoking_mapping': smoking_order,
    'model_name': model_name,
    'random_state': RANDOM_STATE
}

joblib.dump(model_artifact, '../artifacts/models/diabetes_model.pkl')
print('Model saved to artifacts/models/diabetes_model.pkl')

In [ ]:
# Generate predictions on test set
y_test_proba_final = best_model.predict_proba(X_test)[:, 1]
y_test_pred_final = (y_test_proba_final >= selected_threshold).astype(int)

# Create predictions dataframe
predictions_df = pd.DataFrame({
    'actual': y_test.values,
    'predicted': y_test_pred_final,
    'probability': y_test_proba_final
})

# Save predictions
predictions_df.to_csv('../artifacts/predictions/test_predictions.csv', index=False)
print(f'Predictions saved to artifacts/predictions/test_predictions.csv')
print(f'\nPrediction distribution:')
print(predictions_df['predicted'].value_counts())

In [ ]:
# Save final metrics summary
final_metrics = {
    'model': model_name,
    'threshold': selected_threshold,
    'test_roc_auc': final_results['test']['roc_auc'],
    'test_precision': final_results['test']['precision'],
    'test_recall': final_results['test']['recall'],
    'test_f1': final_results['test']['f1'],
    'test_accuracy': final_results['test']['accuracy']
}

with open('../artifacts/models/model_metrics.json', 'w') as f:
    json.dump(final_metrics, f, indent=2)

print('Model metrics saved to artifacts/models/model_metrics.json')
print(json.dumps(final_metrics, indent=2))

In [ ]:
# Save model summary table for reporting
summary_rows = []
for model_name_key, results in all_results.items():
    summary_rows.append({
        'Model': model_name_key,
        'Train ROC-AUC': results['train']['roc_auc'],
        'Val ROC-AUC': results['val']['roc_auc'],
        'Test ROC-AUC': results['test']['roc_auc'],
        'Val F1': results['val']['f1'],
        'Test F1': results['test']['f1'],
        'Test Recall': results['test']['recall']
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv('../reports/tables/model_metrics_summary.csv', index=False)
print('Model metrics summary saved to reports/tables/model_metrics_summary.csv')
print('\n' + summary_df.round(4).to_string(index=False))

## 11. Model Selection Justification

### Final Model: LightGBM

**Why LightGBM?**
1. Achieved highest ROC-AUC on validation and test sets
2. Good balance between precision and recall
3. Fast training and inference
4. Handles class imbalance with `is_unbalance=True`
5. Provides feature importance for interpretability

### Threshold Selection
- Selected threshold based on maximizing F1 while maintaining recall >= 0.60
- This balances the need to catch diabetic patients (recall) while minimizing false alarms (precision)

### Limitations
1. HbA1c_level and blood_glucose_level are clinical measurements that may not be available at initial screening
2. Model is trained on a specific dataset and may not generalize to all populations
3. Correlation does not imply causation
4. Model predictions should support, not replace, clinical judgment